# Исследование датасета и PII-сканера

Notebook фиксирует ход анализа для кейса `ПДнDataset/HACKATHON_CASE.md`: профиль файлового хранилища, запуск CLI-логики на репрезентативном подмножестве и интерпретация уровней защищенности.

## Контекст задачи

Нужно рекурсивно обходить смешанное корпоративное хранилище, извлекать текст из структурированных и документных форматов, находить ПДн, классифицировать категории и формировать отчет. В отчетах не сохраняем полные значения найденных сущностей: используем категории, количества, УЗ и маскированные примеры.

In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path('..').resolve()
DATASET = ROOT / 'ПДнDataset' / 'share'
files = [path for path in DATASET.rglob('*') if path.is_file()]

ext_counts = Counter(path.suffix.lower() or '[no_ext]' for path in files)
ext_counts.most_common()

In [ ]:
size_by_ext = Counter()
for path in files:
    size_by_ext[path.suffix.lower() or '[no_ext]'] += path.stat().st_size

sizes = [
    {'extension': ext, 'mb': round(size / 1024 / 1024, 2)}
    for ext, size in size_by_ext.items()
]
sorted(sizes, key=lambda item: item['mb'], reverse=True)[:20]

## Наблюдения по структуре

Основная нагрузка приходится на PDF, изображения, HTML и офисные документы. Для быстрого прототипа сканер разделен на два слоя: извлечение текста по формату и независимый слой детекторов. OCR отключен по умолчанию, потому что требует системный Tesseract и заметно увеличивает время обработки.

In [ ]:
from pii_scanner.scanner import ScannerConfig, scan_path

billing_dir = DATASET / 'Выгрузки' / 'дочерние предприятия' / 'Billing'
summary = scan_path(
    billing_dir,
    ScannerConfig(only_findings=True),
    workers=2,
    verbose=True,
)
summary['files_scanned'], summary['files_with_pii'], summary['total_findings']

In [ ]:
[
    {
        'path': item['path'],
        'file_format': item['file_format'],
        'protection_level': item['protection_level'],
        'categories': item['categories'],
        'counts': item['counts'],
        'warnings': item['warnings'],
    }
    for item in summary['results']
]

In [ ]:
category_totals = Counter()
for item in summary['results']:
    counts = item['counts']
    category_totals.update(counts)

category_totals.most_common()

## Интерпретация УЗ

- `УЗ-1`: специальные категории или биометрия.
- `УЗ-2`: платежные данные или большой объем государственных идентификаторов.
- `УЗ-3`: государственные идентификаторы в малом объеме или большой объем обычных ПДн.
- `УЗ-4`: только обычные ПДн в небольшом объеме.

Порог большого объема задается параметром `--high-volume-threshold` и по умолчанию равен 100 находкам на файл.

In [ ]:
# Эквивалентный запуск из терминала:
# uv run pii-scan "ПДнDataset/share" \
#   --output reports/pii_report.json \
#   --csv-output reports/pii_report.csv \
#   --markdown-output reports/pii_report.md \
#   --workers 4
